# The Predictive Value of Financial Ratios

## Purpose

A central claim of this study is that filing-behaviour metadata predicts dissolution more effectively than financial statement ratios for the Irish SME population. This notebook tests that claim directly, from two angles:

1. **At register scale.** How does a model built only on financial ratios perform across the full modelling population, and how much does the model lose if financial features are removed entirely?
2. **Where financials actually exist.** Restricting to the companies that have genuine Orbis financial data, do the ratios become competitive once the coverage problem is removed?

To ensure the conclusion does not depend on a single algorithm, each experiment is run across all five candidate models (XGBoost, LightGBM, Logistic Regression, Random Forest, Decision Tree). A finding that holds across every algorithm is a robustness result, not an artefact of one model's inductive bias.

## Why two angles are needed

Financial ratios are sourced from Orbis, which covers a minority of the register. For every other company the financial features are neutral-filled. A register-scale comparison therefore conflates two distinct effects: the intrinsic predictive value of the ratios, and their sparsity. The first experiment measures the population-scale reality; the second isolates the ratios on the subset where they are genuinely observed.

## Method

Every model configuration is independently tuned with Optuna (100 trials, 5-fold stratified cross-validation, Average Precision objective) so that no configuration is disadvantaged by borrowed hyperparameters. All models are evaluated once on the held-out temporal test set. The financial subset comprises the Orbis-derived solvency, liquidity, profitability, decline-trend, and ownership variables present in the modelling data.

This is a substantial computation: five models across three feature sets in Experiment 1, and five models across two feature sets in Experiment 2, each with its own Optuna search. Allow several hours, or reduce `N_TRIALS` for a faster directional pass.

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import xgboost as xgb, lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

ROOT = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import PROCESSED_FILES, RANDOM_STATE, TABLES_DIR

N_TRIALS = 100   # reduce (e.g. 40) for a faster directional run

train = pd.read_csv(PROCESSED_FILES['train_set'], low_memory=False)
test  = pd.read_csv(PROCESSED_FILES['test_set'],  low_memory=False)
print(f'Train {len(train):,} (positive {train["label"].mean()*100:.2f}%) | '
      f'Test {len(test):,} (positive {test["label"].mean()*100:.2f}%)')

Train 98,926 (positive 6.69%) | Test 94,421 (positive 4.07%)


## Feature lists

The full feature set is taken from `models/feature_cols.txt`, the exact list the deployed model uses, so this experiment operates on the same features as the main pipeline. The financial subset is the set of Orbis-derived columns that are actually present in that list; any financial metric not in the modelling data is deliberately excluded rather than assumed.

In [2]:
feat_path = ROOT / 'models' / 'feature_cols.txt'
if feat_path.exists():
    FULL = [l.strip() for l in open(feat_path) if l.strip()]
else:
    non_feat = {'company_num','company_name','company_status','company_type','comp_reg_date',
        'comp_dissolved_date','last_ar_date','last_accounts_date','nace_v2_code',
        'company_address_1','company_address_2','company_address_3','company_address_4','eircode',
        'obs_date','county','nace_sector','company_type_norm','nace_section','nace_section_label',
        'label','dissolution_risk_score','dissolution_risk_tier','if_anomaly_score',
        'if_anomaly_flag','combined_risk_score','combined_risk_tier'}
    FULL = [c for c in train.columns if c not in non_feat and train[c].dtype != object]

# Orbis-derived financial / ownership features, restricted to those actually in FULL
FINANCIAL_CANDIDATES = ['solvency_ratio','current_ratio','total_assets','pl_last_yr',
    'is_insolvent','is_loss_making','illiquid','pl_declining','sol_declining',
    'revenue_declining','revenue_declining_2yr','is_operating_loss','ebit_declining',
    'highly_geared','has_long_term_debt','slow_creditor_payment','employees_declining',
    'gearing_ratio','credit_period','operating_revenue','is_corporate_owned','is_foreign_owned',
    'guo_worldcompliance','guo_irish_company_count','n_subsidiaries_ult']
FIN     = [f for f in FINANCIAL_CANDIDATES if f in FULL]
NON_FIN = [f for f in FULL if f not in FIN]
print(f'FULL {len(FULL)} | FINANCIAL {len(FIN)} | NON-FINANCIAL {len(NON_FIN)}')

FULL 84 | FINANCIAL 20 | NON-FINANCIAL 64


## Model definitions

Each algorithm has its own Optuna search space, mirroring the main model comparison. Tree ensembles receive the raw matrix (XGBoost and LightGBM handle missing values natively; Random Forest and Decision Tree are median-imputed). Logistic Regression is median-imputed and standard-scaled, as a linear model requires. Class imbalance is handled by `scale_pos_weight` for the boosters and `class_weight='balanced'` for the others.

In [3]:
def _impute(a, b):
    imp = SimpleImputer(strategy='median').fit(a); return imp.transform(a), imp.transform(b)
def _scale(a, b):
    sc = StandardScaler().fit(a); return sc.transform(a), sc.transform(b)

MODELS = ['xgb','lgb','lr','rf','dt']
NAMES  = {'xgb':'XGBoost','lgb':'LightGBM','lr':'Logistic Reg','rf':'Random Forest','dt':'Decision Tree'}

def _suggest(mn, trial):
    if mn == 'xgb':
        return dict(n_estimators=trial.suggest_int('n_estimators',200,1000),
            learning_rate=trial.suggest_float('learning_rate',0.01,0.2,log=True),
            max_depth=trial.suggest_int('max_depth',3,9),
            subsample=trial.suggest_float('subsample',0.6,1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree',0.6,1.0),
            reg_alpha=trial.suggest_float('reg_alpha',1e-4,10,log=True),
            reg_lambda=trial.suggest_float('reg_lambda',1e-4,10,log=True))
    if mn == 'lgb':
        return dict(n_estimators=trial.suggest_int('n_estimators',200,1000),
            learning_rate=trial.suggest_float('learning_rate',0.01,0.2,log=True),
            num_leaves=trial.suggest_int('num_leaves',15,255),
            subsample=trial.suggest_float('subsample',0.6,1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree',0.6,1.0))
    if mn == 'lr':
        return dict(C=trial.suggest_float('C',1e-3,100,log=True))
    if mn == 'rf':
        return dict(n_estimators=trial.suggest_int('n_estimators',200,800),
            max_depth=trial.suggest_int('max_depth',5,30),
            min_samples_leaf=trial.suggest_int('min_samples_leaf',1,20))
    if mn == 'dt':
        return dict(max_depth=trial.suggest_int('max_depth',3,25),
            min_samples_leaf=trial.suggest_int('min_samples_leaf',1,50))

def _build(mn, p, spw):
    if mn == 'xgb': return xgb.XGBClassifier(**p, scale_pos_weight=spw, eval_metric='aucpr',
        random_state=RANDOM_STATE, verbosity=0, n_jobs=-1)
    if mn == 'lgb': return lgb.LGBMClassifier(**p, scale_pos_weight=spw,
        random_state=RANDOM_STATE, verbose=-1, n_jobs=-1)
    if mn == 'lr':  return LogisticRegression(**p, class_weight='balanced',
        max_iter=2000, random_state=RANDOM_STATE)
    if mn == 'rf':  return RandomForestClassifier(**p, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1)
    if mn == 'dt':  return DecisionTreeClassifier(**p, class_weight='balanced',
        random_state=RANDOM_STATE)

def run_all_models(features, tr_df, te_df, y_tr, y_te):
    spw = (y_tr==0).sum()/max((y_tr==1).sum(),1)
    Xtr_raw, Xte_raw = tr_df[features].values, te_df[features].values
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    rows = {}
    for mn in MODELS:
        def objective(trial):
            p = _suggest(mn, trial); sc = []
            for t, v in cv.split(Xtr_raw, y_tr):
                xt, xv = Xtr_raw[t], Xtr_raw[v]
                if mn in ('lr','rf','dt'): xt, xv = _impute(xt, xv)
                if mn == 'lr': xt, xv = _scale(xt, xv)
                m = _build(mn, p, spw); m.fit(xt, y_tr[t])
                sc.append(average_precision_score(y_tr[v], m.predict_proba(xv)[:,1]))
            return np.mean(sc)
        study = optuna.create_study(direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
        xt, xte = Xtr_raw, Xte_raw
        if mn in ('lr','rf','dt'): xt, xte = _impute(xt, xte)
        if mn == 'lr': xt, xte = _scale(xt, xte)
        m = _build(mn, study.best_params, spw); m.fit(xt, y_tr)
        p = m.predict_proba(xte)[:,1]
        rows[NAMES[mn]] = (average_precision_score(y_te,p), roc_auc_score(y_te,p))
        print(f'    {NAMES[mn]:<14} AP={rows[NAMES[mn]][0]:.4f}  AUC={rows[NAMES[mn]][1]:.4f}')
    return rows

## Experiment 1 — Register scale

All five models, on the full modelling population, under three feature sets: financials only, the full set, and the full set with financials removed.

In [4]:
y_tr = train['label'].values.astype(int)
y_te = test['label'].values.astype(int)

print('FINANCIALS ONLY'); fin1  = run_all_models(FIN, train, test, y_tr, y_te)
print('FULL MODEL');      full1 = run_all_models(FULL, train, test, y_tr, y_te)
print('NO FINANCIALS');   nof1  = run_all_models(NON_FIN, train, test, y_tr, y_te)

tab1 = pd.DataFrame({
    'Financials only': {k:v[0] for k,v in fin1.items()},
    'Full model':      {k:v[0] for k,v in full1.items()},
    'No financials':   {k:v[0] for k,v in nof1.items()},
}).round(4)
print('\nExperiment 1 - Average Precision by model and feature set:')
print(tab1.to_string())

# Persist Experiment 1 results to disk for the dissertation chapter.
exp1_rows = []
for fset_name, results, n_feat in [
    ('Financials only', fin1,  len(FIN)),
    ('Full model',      full1, len(FULL)),
    ('No financials',   nof1,  len(NON_FIN)),
]:
    for model_name, (ap, auc) in results.items():
        exp1_rows.append({
            'experiment':       1,
            'population':       'register-scale',
            'feature_set':      fset_name,
            'n_features':       n_feat,
            'model':            model_name,
            'avg_precision':    round(ap, 4),
            'auc_roc':          round(auc, 4),
            'n_train':          len(train),
            'n_test':           len(test),
            'train_base_rate':  round(float(train['label'].mean()), 4),
            'test_base_rate':   round(float(test['label'].mean()),  4),
        })
exp1_df = pd.DataFrame(exp1_rows)
exp1_df.to_csv(TABLES_DIR / 'ablation_financial_experiment1.csv', index=False)
tab1.to_csv(TABLES_DIR / 'ablation_financial_experiment1_wide.csv')
print(f"\nSaved: ablation_financial_experiment1.csv ({len(exp1_df)} rows, long format)")
print(f"Saved: ablation_financial_experiment1_wide.csv (pivot for dissertation table)")


FINANCIALS ONLY
    XGBoost        AP=0.0577  AUC=0.5730
    LightGBM       AP=0.0595  AUC=0.5743
    Logistic Reg   AP=0.0483  AUC=0.5624
    Random Forest  AP=0.0574  AUC=0.5753
    Decision Tree  AP=0.0509  AUC=0.5579
FULL MODEL
    XGBoost        AP=0.6333  AUC=0.9432
    LightGBM       AP=0.6273  AUC=0.9387
    Logistic Reg   AP=0.4755  AUC=0.9615
    Random Forest  AP=0.4068  AUC=0.8730
    Decision Tree  AP=0.1789  AUC=0.7840
NO FINANCIALS
    XGBoost        AP=0.6110  AUC=0.9366
    LightGBM       AP=0.6054  AUC=0.9329
    Logistic Reg   AP=0.4585  AUC=0.9578
    Random Forest  AP=0.4042  AUC=0.8676
    Decision Tree  AP=0.1634  AUC=0.7744

Experiment 1 - Average Precision by model and feature set:
               Financials only  Full model  No financials
XGBoost                 0.0577      0.6333         0.6110
LightGBM                0.0595      0.6273         0.6054
Logistic Reg            0.0483      0.4755         0.4585
Random Forest           0.0574      0.4068         0

## Experiment 2 — Covered subset (like-for-like)

Restricting to companies with genuine Orbis data, so financials-only and the full set are compared on the *same* population. A company is treated as covered if its solvency or current ratio is non-zero. The dissolved count in each split is printed first: if it is large the five-model table is reliable; if it were small, only the deployed model's row would be trusted.

The covered subset skews toward larger, established companies that dissolve less often, so its absolute AP is not directly comparable to Experiment 1 and is reported as a separate diagnostic.

In [5]:
def covered(df):
    s = pd.to_numeric(df['solvency_ratio'], errors='coerce').fillna(0)
    c = pd.to_numeric(df['current_ratio'],  errors='coerce').fillna(0)
    return (s != 0) | (c != 0)

tr_c, te_c = train[covered(train)].copy(), test[covered(test)].copy()
print(f'Covered train: {len(tr_c):,} ({len(tr_c)/len(train)*100:.1f}%), '
      f'dissolved {int(tr_c["label"].sum())} ({tr_c["label"].mean()*100:.2f}%)')
print(f'Covered test:  {len(te_c):,} ({len(te_c)/len(test)*100:.1f}%), '
      f'dissolved {int(te_c["label"].sum())} ({te_c["label"].mean()*100:.2f}%)')
reliable = (tr_c['label'].sum() >= 50) and (te_c['label'].sum() >= 30)
print('Five-model table reliable on this subset:', reliable,
      '' if reliable else '(too few failures - read XGBoost row only)')

Covered train: 14,616 (14.8%), dissolved 811 (5.55%)
Covered test:  21,857 (23.1%), dissolved 673 (3.08%)
Five-model table reliable on this subset: True 


In [6]:
yc_tr = tr_c['label'].values.astype(int)
yc_te = te_c['label'].values.astype(int)

print('COVERED SUBSET - FINANCIALS ONLY'); finc  = run_all_models(FIN, tr_c, te_c, yc_tr, yc_te)
print('COVERED SUBSET - FULL SET');        fullc = run_all_models(FULL, tr_c, te_c, yc_tr, yc_te)

tab2 = pd.DataFrame({
    'Financials only': {k:v[0] for k,v in finc.items()},
    'Full set':        {k:v[0] for k,v in fullc.items()},
}).round(4)
print(f'\nExperiment 2 - AP by model on covered subset (base rate {te_c["label"].mean()*100:.2f}%):')
print(tab2.to_string())

# Persist Experiment 2 results to disk for the dissertation chapter.
exp2_rows = []
for fset_name, results, n_feat in [
    ('Financials only', finc,  len(FIN)),
    ('Full set',        fullc, len(FULL)),
]:
    for model_name, (ap, auc) in results.items():
        exp2_rows.append({
            'experiment':       2,
            'population':       'covered-subset',
            'feature_set':      fset_name,
            'n_features':       n_feat,
            'model':            model_name,
            'avg_precision':    round(ap, 4),
            'auc_roc':          round(auc, 4),
            'n_train':          len(tr_c),
            'n_test':           len(te_c),
            'train_base_rate':  round(float(tr_c['label'].mean()), 4),
            'test_base_rate':   round(float(te_c['label'].mean()),  4),
        })
exp2_df = pd.DataFrame(exp2_rows)
exp2_df.to_csv(TABLES_DIR / 'ablation_financial_experiment2.csv', index=False)
tab2.to_csv(TABLES_DIR / 'ablation_financial_experiment2_wide.csv')
print(f"\nSaved: ablation_financial_experiment2.csv ({len(exp2_df)} rows, long format)")
print(f"Saved: ablation_financial_experiment2_wide.csv (pivot for dissertation table)")


COVERED SUBSET - FINANCIALS ONLY
    XGBoost        AP=0.1180  AUC=0.7473
    LightGBM       AP=0.1298  AUC=0.7384
    Logistic Reg   AP=0.0616  AUC=0.7013
    Random Forest  AP=0.1185  AUC=0.7493
    Decision Tree  AP=0.0804  AUC=0.6888
COVERED SUBSET - FULL SET
    XGBoost        AP=0.5147  AUC=0.9525
    LightGBM       AP=0.4821  AUC=0.9448
    Logistic Reg   AP=0.2510  AUC=0.9325
    Random Forest  AP=0.5099  AUC=0.9322
    Decision Tree  AP=0.3008  AUC=0.8505

Experiment 2 - AP by model on covered subset (base rate 3.08%):
               Financials only  Full set
XGBoost                 0.1180    0.5147
LightGBM                0.1298    0.4821
Logistic Reg            0.0616    0.2510
Random Forest           0.1185    0.5099
Decision Tree           0.0804    0.3008

Saved: ablation_financial_experiment2.csv (10 rows, long format)
Saved: ablation_financial_experiment2_wide.csv (pivot for dissertation table)


## Interpretation

**Experiment 1** establishes the population-scale reality across every algorithm: if the financials-only column is uniformly low and the no-financials column is close to the full model, then financial ratios contribute little at register scale, regardless of model choice.

**Experiment 2** clarifies why, by comparing both feature sets on the identical subset of companies that do have financial data. If the non-financial features still lead here, the weakness of financials is not merely a coverage artefact; if financials close the gap on this subset, the register-scale weakness is principally a coverage story.

Because both experiments span all five algorithms, the resulting conclusion is a robustness statement rather than a property of a single model: for the Irish SME population, the predictive signal lies overwhelmingly in filing behaviour rather than in financial ratios, and this holds whichever algorithm is used. This is the empirical foundation for the study's central design choice.